# Electricity Forecasting Application
## 01 — Exploratory Data Analysis (EDA)

### Objective
Explore the UCI Electricity Load Diagrams 2011–2014 dataset, focusing on electricity meter MT_001.

### Dataset
- Source: UCI Machine Learning Repository
- Measurements: 15-minute intervals
- Target meter: MT_001
- Forecasting objective: Predict the next measurement using the previous 96 observations (24 hours).

### Analysis steps
1. Load and inspect the dataset.
2. Analyze missing values and zeros.
3. Study the temporal distribution.
4. Visualize electricity consumption.
5. Identify preprocessing requirements.

**Note:** Initial data cleaning and exploratory analysis are performed in this notebook. Further preprocessing for model training, including normalization, chronological data splitting, and sequence generation, will be performed in separate notebooks.

### 1. Data Loading & Inspection

This step loads the electricity consumption data from all **370 meters** into a Pandas DataFrame.

The objective is to verify the dataset structure (**140,256 observations and 371 columns**) and prepare the data for cleaning, correlation analysis, and clustering.

In [ ]:
import pandas as pd
from pathlib import Path

file_path = Path("../data/raw/LD2011_2014.txt")
# Chargement de tous les compteurs électriques
df_all = pd.read_csv(
    file_path,
    sep=";",
    decimal=",",
    parse_dates=[0]
)

# Vérification des dimensions et des premières lignes
print("Dimensions :", df_all.shape)
display(df_all.head())

### 2. Data Cleaning — Handling Initial and Trailing Zeros

This step identifies and replaces inactive periods (leading and trailing zeros) with `NaN` for all 370 electricity meters, while preserving valid zeros within active periods.

The goal is to improve data quality and ensure reliable correlation analysis and clustering without losing temporal alignment between meters.

In [ ]:
import numpy as np

# Copie des données pour préserver le DataFrame original
df_clean = df_all.copy()

# Sélection des 370 colonnes de compteurs
meter_cols = [col for col in df_clean.columns if col.startswith("MT_")]

# Parcours de chaque compteur
for col in meter_cols:
    serie = df_clean[col]

    # Identifier les positions où la consommation est non nulle
    positions = np.flatnonzero(serie.notna() & (serie != 0))

    # Ignorer les compteurs entièrement nuls
    if len(positions) == 0:
        df_clean[col] = np.nan
        continue

    # Première et dernière observations non nulles
    debut = positions[0]
    fin = positions[-1]

    # Exclure uniquement les zéros en début et fin de série
    df_clean.loc[df_clean.index[:debut], col] = np.nan
    df_clean.loc[df_clean.index[fin + 1:], col] = np.nan

# Vérification du nettoyage
print("Dimensions :", df_clean.shape)
print("Valeurs NaN après nettoyage :",
      df_clean[meter_cols].isna().sum().sum())

### 2.1. Data Cleaning Validation

This step verifies the cleaning results on five selected meters (`MT_001`, `MT_025`, `MT_100`, `MT_200`, `MT_370`).

**Meter Selection:** The five meters were manually selected from different positions in the dataset to check the cleaning process across multiple time series. This is not a random or statistically representative selection.

**Objective:** Confirm that inactive periods were replaced with `NaN` while preserving valid zeros within active periods, ensuring data quality for correlation analysis and clustering.


In [ ]:
# Compteurs sélectionnés pour vérifier le nettoyage
compteurs = ["MT_001", "MT_025", "MT_100", "MT_200", "MT_370"]

resultats = []

for col in compteurs:
    serie = df_clean[col]

    # Identifier les premières et dernières observations valides
    debut = serie.first_valid_index()
    fin = serie.last_valid_index()

    # Rassembler les résultats du contrôle
    resultats.append({
        "Compteur": col,
        "Première position": debut,
        "Dernière position": fin,
        "NaN": serie.isna().sum(),
        "Zéros conservés": (serie == 0).sum()
    })

# Afficher le tableau récapitulatif
display(pd.DataFrame(resultats))

### 3. Correlation Analysis

This step calculates the **Pearson correlation coefficients** between all 370 electricity meters using cleaned data.

**Objectives:**
- Measure the similarity between electricity consumption patterns.
- Identify strongly correlated meters.
- Count overlapping valid observations to assess correlation reliability.
- Visualize the correlation matrix using a **heatmap**.

**Expected Analysis Outputs:**
1. **Correlation Matrix (370 × 370):** Pearson coefficients ranging from −1 to +1, showing relationships between meters.
2. **Overlap Matrix (370 × 370):** Number of valid observations shared by each pair of meters.
3. **Correlation Heatmap:** Visual representation highlighting strongly correlated groups and differences in consumption behavior.
4. **Potential Cluster Identification:** Identification of candidate groups of meters with similar temporal patterns, without determining the final number of clusters.

**Purpose:** These results provide the foundation for clustering electricity meters and training specialized LSTM models for each group.

In [ ]:
# Calcul des corrélations entre les 370 compteurs
corr_matrix = df_clean[meter_cols].corr(method="pearson")

# Nombre d'observations communes valides entre chaque paire
valid = df_clean[meter_cols].notna().astype("int32")
overlap_matrix = valid.T @ valid

# Vérification des dimensions
print("Matrice de corrélation :", corr_matrix.shape)
print("Matrice des observations communes :", overlap_matrix.shape)

# Aperçu des corrélations entre les cinq premiers compteurs
display(corr_matrix.iloc[:5, :5].round(3))

In [ ]:
import matplotlib.pyplot as plt

# Création de la figure
plt.figure(figsize=(14, 12))

# Affichage de la matrice de corrélation
plt.imshow(
    corr_matrix,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    interpolation="nearest"
)

# Échelle des coefficients de corrélation
plt.colorbar(label="Corrélation de Pearson")

# Titres et axes
plt.title("Matrice de corrélation — 370 compteurs")
plt.xlabel("Compteurs électriques")
plt.ylabel("Compteurs électriques")

plt.tight_layout()
plt.show()